*** This code is to test the windy environments ***

In [1]:
# Import libraries
import pygame
import gymnasium as gym
import numpy as np
import copy
import itertools
import math
np.random.seed(33) # seeding

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# convert binary list to decimal
def binary_list_to_decimal(bin_list):
    bin = ''
    for b in bin_list:
        bin += str(b)
    dec = int(bin,2)
    return dec

# Function to check if a point is inside a polygon (Ray-casting algorithm)
def is_inside_polygon(point, poly):
    x, y = point
    inside = False
    n = len(poly)
    p1x, p1y = poly[0]
    for i in range(n + 1):
        p2x, p2y = poly[i % n]
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
            if p1x == p2x or x <= xinters:
                inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Function to return minimum distance in a list of points
def min_dist(x):
    x = np.array(x).astype('float32')
    dists = []
    for p1, p2 in itertools.combinations(x, 2):
        dist = np.linalg.norm(p1-p2)
        dists.append(dist)
    return float(np.min(dists))

In [3]:
experiments_path = r'./experiment_sets.txt'
# Read the experiments file and select the experiment
with open(experiments_path, 'r') as experiment_file:
    codes = experiment_file.read()
    exec(codes) # execute
selected_experiment = set3 # Select the experiment set
selected_experiment

{'field': [(13.0, 23.0), (27.0, 21.0), (31.0, 32.0), (27.0, 33.0)],
 'init_positions': [array([20., 25.]), array([30., 30.]), array([25., 30.])],
 'infected_locations': {(16.0, 25.0),
  (20.0, 24.0),
  (21.0, 28.0),
  (24.0, 30.0),
  (27.0, 23.0),
  (27.0, 28.0)}}

In [4]:
sf = 10 # scaling factor
selected_experiment['field'] = [(x*sf, y*sf) for (x,y) in selected_experiment['field']]
selected_experiment['infected_locations'] = [(x*sf, y*sf) for (x,y) in selected_experiment['infected_locations']]
selected_experiment['init_positions'] = [v*sf for v in selected_experiment['init_positions']]
selected_experiment, len(selected_experiment['init_positions']), len(np.unique(selected_experiment['init_positions'], axis=0))

({'field': [(130.0, 230.0), (270.0, 210.0), (310.0, 320.0), (270.0, 330.0)],
  'init_positions': [array([200., 250.]),
   array([300., 300.]),
   array([250., 300.])],
  'infected_locations': [(240.0, 300.0),
   (160.0, 250.0),
   (200.0, 240.0),
   (270.0, 280.0),
   (210.0, 280.0),
   (270.0, 230.0)]},
 3,
 3)

# Inference

In [5]:
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 4}
    def __init__(self, render_mode=None, poly_vertices=copy.deepcopy(selected_experiment['field']), wind_par=[0,0]):
        super(MultiRobotEnv, self).__init__()

        # Screen dimensions
        self.edge_buffer = 10 # Boundary above the max values
        self.xs, self.ys = zip(*poly_vertices) # x and y values of the vertices of the polygonal field
        self.WIDTH, self.HEIGHT = 1000, 1000 # Use this if we want to have fixed width and height. Default: 800x600        
        # self.WIDTH, self.HEIGHT = max(self.xs) + self.edge_buffer, max(self.ys) + self.edge_buffer
        self.poly_vertices = poly_vertices # Vertices of polygon

        # Number of robots
        self.num_robots = 3 # Rendering error if more than 7

        # Robot parameters
        self.robot_size = 10
        self.mass = 1.0
        # self.g = 0.1  # Gravity or directional force
        self.thrust_power = 0.5  # Force applied per action
        self.max_speed = 5  # Maximum speed    
        self.min_speed = -5 # Minimum speed
        self.min_positions = np.zeros(self.num_robots*2) # Minimum positions
        self.max_positions = np.array([[self.WIDTH, self.HEIGHT] for _ in range(self.num_robots)]) # Maximum positions
        self.min_velocities = np.array([[self.min_speed, self.min_speed] for _ in range(self.num_robots)]) # Min speed list
        self.max_velocities = np.array([[self.max_speed, self.max_speed] for _ in range(self.num_robots)]) # Max speed list
        self.wind_f_a, self.wind_beta_a = wind_par

        # infected locations
        self.infected_size = 10 # Radius of infected locations
        self.infected_length = len(copy.deepcopy(selected_experiment['infected_locations']))
        self.infected_state_length = 2**(self.infected_length) # 2**5, binary to decimal

        # Action space: thrust in x and y directions for each robot
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(self.num_robots, 2), dtype=np.float32)

        # Observation space: position and velocity (x, y, vx, vy) for each robot + infected location        
        self.observation_space = gym.spaces.Box( # The (visited) weed locations are tracked on the observation space
                    low = np.concatenate((self.min_positions.flatten(), self.min_velocities.flatten(), np.array([0]))), # Lowest positions and velocities
                    high = np.concatenate((self.max_positions.flatten(), self.max_velocities.flatten(), np.array([self.infected_state_length - 1]))), # highest positions and velocities
                    dtype=np.float32)

        assert render_mode is None or render_mode in self.metadata["render_modes"] # Check if the render mode is correct
        self.render_mode = render_mode
        self.screen = None
        self.clock = None
        # If human-rendering is used, `self.screen` will be a reference to the screen that we draw to. `self.clock` will be a clock that is used
        # to ensure that the environment is rendered at the correct framerate in human-mode. They will remain `None` until human-mode is used for the first time.   

        # Reset the environment and start
        self.reset()
    
    def _get_obs(self):
        info = {f'robot{i}': self.robot_positions[i] for i in range(self.num_robots)} # Current position of each robot
        infected = binary_list_to_decimal(list(self.infected_dict.values())) # Convert the binary list of infected locations to a decimal value
        state = np.concatenate((self.robot_positions.flatten(), self.robot_velocities.flatten(), np.array([infected])), dtype=np.float32) # Current state of the robots
        return state, info        

    def reset(self, seed=None, options={}):
        # Reset the visited states and counts
        self.step_count = 0
        self.visited = set() # Keep track of the visited states in an episode
        self.infected_locations = copy.deepcopy(selected_experiment['infected_locations'])
        self.infected_dict = {v:0 for v in self.infected_locations} # 0 for unvisited infected locations, 1 for visited
        self.robot_positions = np.array(copy.deepcopy(selected_experiment['init_positions']))[:self.num_robots] # Initial positions of each robot
        self.robot_velocities = np.zeros((self.num_robots, 2)) # Initial velocities of each robot (zero)
        return self._get_obs()
    
    def step(self, actions):
        terminated, truncated = False, False
        rewards = 0
        self.step_count += 1
        for i in range(self.num_robots): # For every robot
            ax, ay = actions[i] * self.thrust_power # What actions to take

            # Update velocity
            self.robot_velocities[i][0] += ax / self.mass + self.wind_f_a * math.cos(math.radians(self.wind_beta_a))
            self.robot_velocities[i][1] += ay / self.mass + self.wind_f_a * math.sin(math.radians(self.wind_beta_a))

            # Limit velocity
            self.robot_velocities[i] = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Predict new position
            new_position = self.robot_positions[i] + self.robot_velocities[i]

            # Boundary conditions (keep robot within polygon)
            if is_inside_polygon(new_position, self.poly_vertices):
                pass
            else: # Hits the wall!
                rewards -= 10 # Medium negative reward for hitting the wall
                self.robot_velocities[i][:] = 0 # Stop movement

            # Update position
            self.robot_positions[i] += self.robot_velocities[i]
            
            # Boundary conditions (keep robot within screen)
            self.robot_positions[i] = np.clip(self.robot_positions[i], [0, 0], [self.WIDTH, self.HEIGHT]) # Optinal checking to see if the robot is within the Pygame window

            # Check if location is visited before, and add it to the visited locations
            if tuple(self.robot_positions[i]) in self.visited:
                rewards -= 10 # Small negative reward for visiting previous location
            else:
                rewards -= 1 # Very small negative reward for visiting new locations
            self.visited.add(tuple(self.robot_positions[i]))            

            # Check if any infected location is visited        
            nearby_infected_locations = [] # To store the nearby infected locations
            for j, inf_loc in enumerate(self.infected_locations): # Loop through each infected location
                dist = np.linalg.norm(self.robot_positions[i]-inf_loc) # Distance between robot position and infected location
                if dist <= self.infected_size: # If the distance is within the radius of the infected location size
                    nearby_infected_locations.append(inf_loc) # Add the infected location
                    rewards += 100 # Medium positive rewards for visiting each infected location
                    # input("Pause!") # Only pause if you want to visualize visiting infected locations
            for inf_loc in nearby_infected_locations:
                self.infected_locations.remove(inf_loc) # Delete each visited infected location
                self.infected_dict[tuple(inf_loc)] = 1 # Update the infected dictionary
        
        # Check if all infected locations are visited
        if len(self.infected_locations) == 0:
            rewards += 100000 # Big positive rewards for visiting all infected locations
            terminated = True
        
        # Check if any collisions occurred
        if self.num_robots > 1:
            min_dist_between_robots = min_dist(self.robot_positions) # Minimum distance between robots
            if min_dist_between_robots < self.robot_size:
                rewards -= 100000 # Big negative rewards for collisions
                terminated = True

        obs, info = self._get_obs() # Get the updated observations
        # rewards = rewards * self.gamma ** self.step_count
        return obs, rewards, terminated, truncated, info
    
    def render(self):
        # Initialize pygame
        if self.screen is None and self.render_mode == "human": # Initialize pygame if it is not initialized
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
            pygame.display.set_caption("Multi-robot RL Environment")
            if self.clock is None:
                self.clock = pygame.time.Clock()
                self.running = True
        
        self.screen.fill((255, 255, 255)) # White color for the background
        colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128)]  # Colors for each robot: Red, Green, Blue, Orange, Violet, Pink, Grey
        pix_size = 10

        # Draw the polygon
        # pixel_poly_vertices = [(point[0] * pix_size, point[1] * pix_size) for point in self.poly_vertices]
        pygame.draw.polygon(surface=self.screen, 
                            color=(255, 255, 0), # Yello color for the polygon
                            points=self.poly_vertices)
        
        # Draw the visited regions
        for point in self.visited:
            pygame.draw.circle(self.screen, pygame.Color(100, 100, 100, a=0.2), point, pix_size/2) # Light grey color for visited regions, with transparency alpha

        # Draw robots
        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, colors[i], (int(self.robot_positions[i][0]), int(self.robot_positions[i][1])), pix_size/2) # Pick the colors from above list

        # Draw infected locations
            for l in self.infected_locations:
                pygame.draw.circle(self.screen, (0, 255, 255), (int(l[0]), int(l[1])), pix_size/2) # Cyan color for infected locations
        
        pygame.display.flip() # Allows only a portion of the screen to be updated
        self.clock.tick(60)
    
    def close(self):
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()

In [6]:
# Register environment
gym.register(id='MultiRobotEnv-v0', 
             entry_point=MultiRobotEnv,
             max_episode_steps=1000)

Load trained network:

In [7]:
# from sb3_contrib import TRPO

# weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\github\FlowBotic\trained_models\new_mar25_env1_trpo.zip"

# # Load trained network
# model = TRPO.load(weights_path)

In [8]:
from sb3_contrib import CrossQ

weights_path = rf"C:\Users\choton\OneDrive - Kansas State University\PhD Projects\Reinforcement Learning\Codes\for_coRL\github\FlowBotic\trained_models\new_mar25_env3_CrossQ.zip"

# Load trained network
model = CrossQ.load(weights_path)

Play using trained network and default env (we can also use vector env):

In [9]:
def play(wind_par):
    # Make the environment
    env = gym.make('MultiRobotEnv-v0', render_mode='human', wind_par=wind_par)
    env.metadata['render_fps'] = 1
    obs, info = env.reset()
    env.render()
    pygame.event.get()

    # Start playing
    terminated, truncated = False, False
    total_rewards = 0
    total_steps = 0
    while True:
        action, _ = model.predict(obs)
        print(action)
        # print(int(action))
        obs, reward, terminated, truncated,  info = env.step(action)
        env.render()
        total_rewards += reward
        print(f"Obs: {obs}, Reward: {reward}, terminated: {terminated}, total_rewards: {total_rewards}, total_steps: {total_steps}")
        if terminated or truncated:
            print('terminated:', terminated, 'truncated:', truncated)
            break
        pygame.event.get()
        total_steps += 1

In [10]:
assert False, "Play one by one"

AssertionError: Play one by one

In [11]:
play(wind_par=[0,0])

c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


[[ 0.9714471  -0.8893926 ]
 [-0.19152325 -0.0495441 ]
 [-0.9944102   0.80674195]]
Obs: [ 2.0048572e+02  2.4955530e+02  2.9990424e+02  2.9997522e+02
  2.4950279e+02  3.0040338e+02  4.8572356e-01 -4.4469631e-01
 -9.5761627e-02 -2.4772048e-02 -4.9720511e-01  4.0337098e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.9282105  -0.8283657 ]
 [-0.7969734  -0.88270843]
 [-0.99670637 -0.9889319 ]]
Obs: [ 2.0143555e+02  2.4869643e+02  2.9941000e+02  2.9950909e+02
  2.4850723e+02  3.0031229e+02  9.4982880e-01 -8.5887915e-01
 -4.9424833e-01 -4.6612626e-01 -9.9555826e-01 -9.1094971e-02
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[ 0.9695983  -0.686965  ]
 [-0.8542555   0.36772215]
 [-0.99414176 -0.96545297]]
Obs: [ 2.0287018e+02  2.4749406e+02  2.9848862e+02  2.9922684e+02
  2.4701460e+02  2.9973846e+02  1.4346280e+00 -1.2023616e+00
 -9.2137611e-01 -2.8226519e-01 -1.4926292e+00 -5.7382143e-01
  4.0000000e+01], Rew

In [14]:
play(wind_par=[0.1,30])

c:\Users\choton\miniconda3\envs\rl4pag\Lib\site-packages\gymnasium\spaces\box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(


[[ 0.9641315  -0.9370274 ]
 [ 0.4952476  -0.21823853]
 [-0.985937    0.8249414 ]]
Obs: [ 2.0056866e+02  2.4958148e+02  3.0033423e+02  2.9994089e+02
  2.4959363e+02  3.0046246e+02  5.6866825e-01 -4.1851369e-01
  3.3422634e-01 -5.9119266e-02 -4.0636596e-01  4.6247071e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.8159585  -0.97845614]
 [-0.90448314 -0.9536918 ]
 [-0.998687   -0.81699   ]]
Obs: [ 2.0163191e+02  2.4872374e+02  3.0030283e+02  2.9945493e+02
  2.4877452e+02  3.0056644e+02  1.0632501e+00 -8.5774171e-01
 -3.1412691e-02 -4.8596513e-01 -8.1910694e-01  1.0397571e-01
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[ 0.97313905 -0.7867175 ]
 [-0.95081264  0.04554546]
 [-0.9996869  -0.6808569 ]]
Obs: [ 2.0326834e+02  2.4752264e+02  2.9988260e+02  2.9904172e+02
  2.4754218e+02  3.0038000e+02  1.6364222e+00 -1.2011005e+00
 -4.2021647e-01 -4.1319242e-01 -1.2323478e+00 -1.8645272e-01
  4.0000000e+01], Rew

In [16]:
play(wind_par=[0.2,30])

[[ 0.9956745  -0.9762027 ]
 [-0.42499614  0.08526397]
 [-0.98876625  0.76262844]]
Obs: [ 2.0067104e+02  2.4961189e+02  2.9996069e+02  3.0014264e+02
  2.4967882e+02  3.0048132e+02  6.7104232e-01 -3.8810137e-01
 -3.9292991e-02  1.4263198e-01 -3.2117805e-01  4.8131421e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.97447276 -0.98470896]
 [-0.93787074 -0.7942541 ]
 [-0.99990046 -0.70185804]]
Obs: [ 2.0200253e+02  2.4883144e+02  2.9962567e+02  2.9998813e+02
  2.4903090e+02  3.0071170e+02  1.3314838e+00 -7.8045583e-01
 -3.3502328e-01 -1.5449509e-01 -6.4792323e-01  2.3038518e-01
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[ 0.98846483 -0.9556732 ]
 [-0.96589506 -0.7781577 ]
 [-0.9980151  -0.9235167 ]]
Obs: [ 2.0400145e+02  2.4767316e+02  2.9898093e+02  2.9954456e+02
  2.4805717e+02  3.0058032e+02  1.9989213e+00 -1.1582925e+00
 -6.4476573e-01 -4.4357395e-01 -9.7372568e-01 -1.3137317e-01
  4.0000000e+01], Rew

In [18]:
play(wind_par=[0.3,30])

[[ 0.98080254 -0.9303776 ]
 [-0.20893991 -0.00347328]
 [-0.9942253   0.6973655 ]]
Obs: [ 2.0075021e+02  2.4968481e+02  3.0015533e+02  3.0014825e+02
  2.4976270e+02  3.0049869e+02  7.5020885e-01 -3.1518880e-01
  1.5533766e-01  1.4826337e-01 -2.3730505e-01  4.9868277e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.27689314 -0.9811692 ]
 [-0.9147068  -0.94604796]
 [-0.9993983  -0.8740901 ]]
Obs: [ 2.0189867e+02  2.4902904e+02  3.0011313e+02  2.9997351e+02
  2.4928549e+02  3.0071033e+02  1.1484630e+00 -6.5577340e-01
 -4.2208135e-02 -1.7476061e-01 -4.7719657e-01  2.1163774e-01
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[ 0.9726794  -0.92333823]
 [-0.94027025 -0.7071294 ]
 [-0.9990426  -0.8961541 ]]
Obs: [ 2.0379329e+02  2.4806160e+02  2.9986060e+02  2.9959518e+02
  2.4856859e+02  3.0062387e+02  1.8946103e+00 -9.6744251e-01
 -2.5253564e-01 -3.7832531e-01 -7.1691024e-01 -8.6439312e-02
  4.0000000e+01], Rew

In [21]:
play(wind_par=[0.4,30])

[[ 0.9600396  -0.74965763]
 [ 0.04250073  0.04585838]
 [-0.99691343  0.73395383]]
Obs: [ 2.0082643e+02  2.4982516e+02  3.0036768e+02  3.0022293e+02
  2.4984795e+02  3.0056699e+02  8.2642996e-01 -1.7482881e-01
  3.6766052e-01  2.2292919e-01 -1.5204656e-01  5.6697690e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.8833685  -0.96089065]
 [-0.6079554  -0.97124016]
 [-0.99920535 -0.9362115 ]]
Obs: [ 2.0244095e+02  2.4936990e+02  3.0077774e+02  3.0016025e+02
  2.4954271e+02  3.0086584e+02  1.6145244e+00 -4.5527416e-01
  4.1009298e-01 -6.2690899e-02 -3.0523908e-01  2.9887116e-01
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[ 0.8422022  -0.81447697]
 [-0.9369975  -0.8847842 ]
 [-0.9981822  -0.8628375 ]]
Obs: [ 2.0482298e+02  2.4870738e+02  3.0106577e+02  2.9985516e+02
  2.4908479e+02  3.0093329e+02  2.3820357e+00 -6.6251266e-01
  2.8800440e-01 -3.0508301e-01 -4.5792001e-01  6.7452416e-02
  4.0000000e+01], Rew

In [22]:
play(wind_par=[0.5,30])

[[ 0.9626353  -0.92596126]
 [-0.25820917 -0.20584589]
 [-0.9947065   0.8878212 ]]
Obs: [ 2.0091434e+02  2.4978702e+02  3.0030389e+02  3.0014706e+02
  2.4993565e+02  3.0069391e+02  9.1433036e-01 -2.1298063e-01
  3.0390811e-01  1.4707705e-01 -6.4340562e-02  6.9391060e-01
  4.0000000e+01], Reward: 197, terminated: False, total_rewards: 197, total_steps: 0
[[ 0.9394026  -0.9893396 ]
 [-0.19964576 -0.98651075]
 [-0.9996469  -0.41748178]]
Obs: [ 2.0273137e+02  2.4932938e+02  3.0094101e+02  3.0005090e+02
  2.4980450e+02  3.0142908e+02  1.8170444e+00 -4.5765042e-01
  6.3709795e-01 -9.6178323e-02 -1.3115132e-01  7.3516971e-01
  4.0000000e+01], Reward: -3, terminated: False, total_rewards: 194, total_steps: 1
[[-0.4199704 -0.991283 ]
 [-0.9259531 -0.9907454]
 [-0.9948433 -0.8778578]]
Obs: [ 2.0477145e+02  2.4862608e+02  3.0154813e+02  2.9970935e+02
  2.4960895e+02  3.0197531e+02  2.0400720e+00 -7.0329189e-01
  6.0713410e-01 -3.4155104e-01 -1.9556028e-01  5.4624081e-01
  4.0000000e+01], Reward: -

**Results**: The learned environment is robust up to the wind speed of 0.4 m/s

In [24]:
env = gym.make('MultiRobotEnv-v0', render_mode='human')
env.metadata['render_fps'] = 1
obs, info = env.reset()
env.render()
pygame.event.get()

[]